# SSM/I **sea-ice concentration** over DBO — streaming, EASE-Grid 2.0 North (no storage)

Sea-ice companion to the chl / SST / cloud notebooks. Streams the **NSIDC passive-microwave (SMMR +
SSM/I-SSMIS, Bootstrap algorithm) daily sea-ice concentration**, grids each day onto the **same
EASE-2 4 km DBO grid**, and folds it into **in-memory accumulators** (nothing written but a
checkpoint). Same through-time analysis (per-box monthly + anomaly + Theil–Sen/AR(1) Mann–Kendall,
seasonal calendar, trend map, GIF), plus the DBO **daily sea-ice + NCEP 10 m winds** figure.

### Product
- **`NSIDC-0079`** (Bootstrap Sea Ice Concentrations from Nimbus-7 SMMR and DMSP SSM/I-SSMIS),
  daily, polar-stereo, 1978→present. v4 (concept_id `C3177837864-NSIDC_CPRD`) is netCDF4 (streamable
  with `h5netcdf`). Swap `ICE_SHORT="NSIDC-0051"` for the NASA Team algorithm, or `G02202` for the CDR.
- Concentration is stored as **fraction (0–1)**; §4 rescales to **% (0–100)** and masks land/pole/missing
  (coded > 1.0). The sensor prefix in the variable name changes over the record
  (`N07_/F08_/F11_/F13_/F17_/F18_ICECON`), so leave `ICE_VAR = None` (auto-detect).
- **SMMR era (1978–1987) sampled every other day** — the off-days are empty stub files (no
  concentration variable); §4 returns them as all-NaN no-obs days. Daily coverage starts ~Aug 1987 (SSM/I).
- Winds: **NCEP/NCAR Reanalysis-1, 10 m, daily-averaged** (from 6-hourly), via NOAA PSL OPeNDAP —
  overlaid as vectors, matching the DBO image.

## 1. Imports

In [1]:
import time as _time, warnings, gc
from pathlib import Path
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

import numpy as np, pandas as pd, xarray as xr
import earthaccess
from pyresample import AreaDefinition, geometry
from pyresample.kd_tree import resample_nearest
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import cartopy.crs as ccrs, cartopy.feature as cfeature
from scipy import stats
from scipy.stats import theilslopes, kendalltau, norm
warnings.filterwarnings("ignore")
print("imports OK")

imports OK


## 2. Configuration — product, EASE-2 grid, DBO boxes, NCEP winds

In [2]:
# >>> confirm with §3 (discovery) and §3b (inspect) <<<
ICE_SHORT   = "NSIDC-0079"          # Bootstrap algorithm (was NSIDC-0051 = NASA Team)
ICE_VERSION = "4"
ICE_CONCEPT = "C3177837864-NSIDC_CPRD"   # Bootstrap v4 (1978 -> ongoing); pins version so v3+v4 don't mix
ICE_VAR     = None                 # None = auto-detect the concentration variable in §4
LATN, LONN  = "latitude", "longitude"

START_DATE = date(1980, 1, 1)
END_DATE   = date.today()
LAT_MIN, LAT_MAX = 45, 90
ROI = 30_000                       # nearest-neighbour radius (m); SSM/I ~25 km
N_WORKERS = 8

dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000))

CHECKPOINT = True
CKPT_PATH  = Path.home() / "ssmi_seaice_ckpt.pkl"  # add your path
CKPT_EVERY = 500
CACHE_DERIVED = False
CACHE_PATH    = Path.home() / "ssmi_seaice_products.pkl"  # add your path

# NCEP/NCAR Reanalysis-1 daily 10 m winds (NOAA PSL OPeNDAP)
NCEP_BASE = "https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis.dailyavgs/surface_gauss"

DBO_BOXES = {
    "DBO1": [(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
    "DBO2": [(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
    "DBO3": [(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
    "DBO4": [(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
    "DBO5": [(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
    "DBO6": [(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
    "DBO7": [(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
    "DBO8": [(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)],
}
data_crs = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=0)
disp     = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=-152)
x0, y0, x1, y1 = dbo_area.area_extent
try: import cmocean; ICE_CMAP = cmocean.cm.ice
except ImportError: ICE_CMAP = "Blues_r"
print(f"{ICE_SHORT} v{ICE_VERSION} | {START_DATE} -> {END_DATE} | grid {dbo_area.shape} | streaming (no disk)")

NSIDC-0079 v4 | 1980-01-01 -> 2026-07-18 | grid (845, 906) | streaming (no disk)


## 3. Auth + discover the product + search granules

In [ ]:
import re
auth = earthaccess.login(persist=True)

# informational discovery — reflects whatever ICE_SHORT is set to (still lists ALL versions)
print(f"=== datasets for short_name={ICE_SHORT!r} ===")
for c in earthaccess.search_datasets(short_name=ICE_SHORT):
    s = c.summary()
    print(f"  short_name={s.get('short-name')!r}  version={s.get('version')!r}  concept_id={s.get('concept-id')!r}")

def _search(s, e, limit=200000):
    kw = ({"concept_id": ICE_CONCEPT} if ICE_CONCEPT
          else {"short_name": ICE_SHORT, **({"version": ICE_VERSION} if ICE_VERSION else {})})
    for k in range(5):
        try: return earthaccess.search_data(temporal=(s, e), count=limit, **kw)
        except Exception as ex: print("  retry:", str(ex)[:80]); _time.sleep(2*(k+1))
    raise RuntimeError("CMR search failed")

def granule_date(g):
    return date.fromisoformat(g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10])

def _is_north_daily(g):
    """NSIDC-0051/0079 bundle N+S hemispheres and daily+monthly in one collection.
    Keep only Northern-hemisphere DAILY files (..._PS_N25km_YYYYMMDD_...) for the DBO grid."""
    fn = g.data_links()[0].rsplit("/", 1)[-1]
    return ("_PS_N25km_" in fn) and bool(re.search(r"_\d{8}_v", fn))

all_granules = [g for g in _search(START_DATE.isoformat(), END_DATE.isoformat()) if _is_north_daily(g)]
granules_by_day = {}
for g in all_granules:
    granules_by_day.setdefault(granule_date(g), g)
print(f"\n{len(all_granules)} N-daily granules | {len(granules_by_day)} unique days "
      f"({min(granules_by_day)} -> {max(granules_by_day)})" if granules_by_day else "no granules")
if all_granules:   # confirm WHICH product actually resolved (NSIDC0079_…=Bootstrap, NSIDC0051_…=NASA Team)
    print("resolved file example:", all_granules[0].data_links()[0].rsplit('/', 1)[-1])

=== datasets for short_name='NSIDC-0079' ===
  short_name='NSIDC-0079'  version='3'  concept_id='C2985422282-NSIDC_CPRD'
  short_name='NSIDC-0079'  version='4'  concept_id='C3177837864-NSIDC_CPRD'


## 4. Single-day reader + gridder (stream → EASE-2)

In [5]:
def _find_var(ds):
    """Pick the ice-concentration variable. Sensor prefix changes over the record
    (N07_/F08_/F11_/F13_/F17_/F18_ICECON, or *_seaice_conc for the CDR). Never returns crs."""
    if ICE_VAR and ICE_VAR in ds.variables: return ICE_VAR
    skip = {"crs", "time", "x", "y", "xgrid", "ygrid", LATN, LONN}
    for v in ds.variables:
        if v in skip: continue
        if v.lower().endswith("icecon") or any(k in v.lower() for k in
               ("seaice_conc", "ice_conc", "icecon", "concentration")):
            return v
    return None

def _read_ice(f):
    with xr.open_dataset(f, engine="h5netcdf") as ds:
        var = _find_var(ds)
        if var is None or ds[var].size == 0:           # SMMR off-day / empty stub -> no obs
            return None
        ci  = np.asarray(ds[var].values, float).squeeze()
        if LATN in ds and LONN in ds:                       # geolocation present -> use it directly
            lat = np.asarray(ds[LATN].values, float); lon = np.asarray(ds[LONN].values, float)
        else:                                               # only x/y projection coords -> derive lon/lat ONCE
            import pyproj
            xc = np.asarray((ds["x"] if "x" in ds else ds["xgrid"]).values, float)
            yc = np.asarray((ds["y"] if "y" in ds else ds["ygrid"]).values, float)
            X, Y = np.meshgrid(xc, yc)
            lon, lat = pyproj.Transformer.from_crs("EPSG:3411", "EPSG:4326", always_xy=True).transform(X, Y)
    if np.nanmax(ci) <= 1.5:           # fraction -> percent
        ci = ci * 100.0
    ci = np.where((ci >= 0) & (ci <= 100), ci, np.nan)      # mask flags (land/pole/missing > 100)
    if lat.ndim == 1 and ci.ndim == 2:                      # 1-D coords -> 2-D geolocation
        lon, lat = np.meshgrid(lon, lat)
    return lat, lon, ci

def ice_to_grid(lat, lon, ci, area):
    lon = np.where(lon > 180, lon - 360, lon)
    m = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(ci)
    swath = geometry.SwathDefinition(lons=lon[m], lats=lat[m])
    grid = resample_nearest(swath, ci[m], area, radius_of_influence=ROI, fill_value=np.nan, reduce_data=False)
    return grid.astype(np.float32)

def project_day(granule, area):
    f = earthaccess.open([granule], show_progress=False)[0]
    try: out = _read_ice(f)
    finally:
        try: f.close()
        except Exception: pass
    if out is None:                                    # empty/no-data day (e.g. SMMR off-day) -> all-NaN
        return np.full(area.shape, np.nan, np.float32)
    return ice_to_grid(*out, area)

## 5. DBO box masks on the EASE-2 grid

In [6]:
lons2d, lats2d = dbo_area.get_lonlats()
grid_pts = np.column_stack([lons2d.ravel(), lats2d.ravel()])
box_masks = {name: MplPath([(lo,la) for la,lo in corners] + [(corners[0][1], corners[0][0])])
                       .contains_points(grid_pts).reshape(lons2d.shape)
             for name, corners in DBO_BOXES.items()}
for name, m in box_masks.items():
    print(f"  {name}: {int(m.sum()):>6,} grid cells")

  DBO1:  2,412 grid cells
  DBO2:    605 grid cells
  DBO3:  2,561 grid cells
  DBO4:  1,293 grid cells
  DBO5:    351 grid cells
  DBO6:    739 grid cells
  DBO7:    773 grid cells
  DBO8:    760 grid cells


In [7]:
# === DBO sea-ice edge (15% SIC): decadal + 2012 & current-year — RESUMABLE checkpoints =====
import matplotlib as mpl, pickle
from pathlib import Path
from scipy.ndimage import gaussian_filter
from matplotlib.lines import Line2D

mpl.rcParams.update({"font.family":"sans-serif",
    "font.sans-serif":["Helvetica","Helvetica Neue","Arial","Nimbus Sans","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.6,"axes.edgecolor":"0.2",
    "axes.spines.top":False,"axes.spines.right":False,"legend.frameon":False,
    "savefig.dpi":300,"figure.dpi":120})

EDGE = 15.0
MONTHS = {"March": 3, "September": 9}
DECADES = [(1980,1989),(1990,1999),(2000,2009),(2010,2019),(2020,2025)]
DEC_COLORS = plt.cm.YlGn(np.linspace(0.35, 1.0, len(DECADES)))
CUR_YEAR = max(granules_by_day).year
REF = [(2012, "#111111", "2012 (record-low summer)"),
       (CUR_YEAR, "#e11d1d", f"{CUR_YEAR} (this year so far)")]
ny, nx = dbo_area.shape

EDGE_CACHE = Path.home() / "dbo_edge_maps.pkl"        # final composites (fast reload)  # add your path
DEC_CKPT   = Path.home() / "dbo_edge_decadal_ckpt.pkl"  # resumable partial progress  # add your path
REF_CKPT   = Path.home() / "dbo_edge_ref_ckpt.pkl"  # add your path
DAY_STRIDE = 1                                          # 2 or 3 = fewer days, ~2-3x faster

def _compose(days, keyfn, tag, ckpt, save_every=200):
    days = days[::DAY_STRIDE]
    acc, done = {}, set()
    if ckpt.exists():
        st = pickle.load(open(ckpt, "rb")); acc, done = st["acc"], st["done"]
        print(f"  {tag}: resumed {len(done)} days already done")
    todo = [(d, g) for d, g in days if d not in done]
    print(f"  {tag}: {len(todo)} days to stream")
    def _one(it):
        d, g = it
        try: return d, project_day(g, dbo_area)
        except Exception: return d, None
    def _save():
        tmp = ckpt.with_suffix(".tmp")
        pickle.dump({"acc": acc, "done": done}, open(tmp, "wb")); tmp.replace(ckpt)
    n = 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for d, arr in ex.map(_one, todo):
            n += 1; done.add(d)
            if arr is not None:
                s = acc.setdefault((keyfn(d.year), d.month),
                                   [np.zeros((ny,nx),np.float32), np.zeros((ny,nx),np.float32)])
                m = np.isfinite(arr); s[0][m] += arr[m]; s[1][m] += 1
            if n % save_every == 0: print(f"    {tag} {n}/{len(todo)}"); _save()
    _save()
    return {k: np.where(v[1] > 0, v[0]/v[1], np.nan) for k, v in acc.items()}

def _dec(y): return next((d0 for d0,d1 in DECADES if d0 <= y <= d1), None)

if EDGE_CACHE.exists():
    _c = pickle.load(open(EDGE_CACHE, "rb"))
    dec_maps, ref_maps = _c["dec_maps"], _c["ref_maps"]
    print(f"loaded composites from {EDGE_CACHE.name} — no streaming")
else:
    if "edge_maps" in globals():
        dec_maps = edge_maps; print("reusing edge_maps from §13")
    else:
        dd = [(d,g) for d,g in sorted(granules_by_day.items())
              if d.month in MONTHS.values() and _dec(d.year) is not None]
        dec_maps = _compose(dd, _dec, "decadal", DEC_CKPT)
    ref_years = [y for y,_,_ in REF]
    rd = [(d,g) for d,g in sorted(granules_by_day.items())
          if d.month in MONTHS.values() and d.year in ref_years]
    ref_maps = _compose(rd, lambda y: y, "ref", REF_CKPT)
    pickle.dump({"dec_maps": dec_maps, "ref_maps": ref_maps}, open(EDGE_CACHE, "wb"))
    print(f"saved -> {EDGE_CACHE}")

def _smooth(a, sig=1.5):
    v = np.isfinite(a).astype(float); a0 = np.where(np.isfinite(a), a, 0.0)
    den = gaussian_filter(v, sig); out = gaussian_filter(a0, sig) / np.where(den > 0, den, np.nan)
    out[v == 0] = np.nan; return out

X2, Y2 = np.meshgrid(*dbo_area.get_proj_vectors())
fig, ax = plt.subplots(figsize=(11, 9), subplot_kw={"projection": disp})
ax.set_extent([-180, -120, 54, 76], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2); ax.coastlines(lw=0.4, color="0.4", zorder=3)
ax.gridlines(draw_labels=True, lw=0.3, color="0.7", alpha=0.5)
for ci, (d0, d1) in enumerate(DECADES):
    for mo, ls in [("March","-"), ("September","--")]:
        em = dec_maps.get((d0, MONTHS[mo]))
        if em is None or np.isfinite(em).sum() < 10: continue
        ax.contour(X2, Y2, _smooth(em), levels=[EDGE], colors=[DEC_COLORS[ci]],
                   linewidths=1.4, linestyles=ls, transform=data_crs, zorder=4)
for y, color, lab in REF:
    drew = False
    for mo, ls in [("March","-"), ("September","--")]:
        em = ref_maps.get((y, MONTHS[mo]))
        if em is None or np.isfinite(em).sum() < 10: continue
        ax.contour(X2, Y2, _smooth(em), levels=[EDGE], colors=[color],
                   linewidths=2.8, linestyles=ls, transform=data_crs, zorder=6)
        drew = True
    if not drew: print(f"  note: no March/September edge available yet for {y}")
for name, corners in DBO_BOXES.items():
    ring = corners + [corners[0]]
    ax.plot([p[1] for p in ring], [p[0] for p in ring], transform=ccrs.PlateCarree(),
            color="#c1272d", lw=1.2, zorder=5)
    cla = np.mean([c[0] for c in corners]); clo = np.mean([c[1] for c in corners])
    ax.text(clo, cla, name[-1], transform=ccrs.PlateCarree(), color="#c1272d", fontsize=9,
            fontweight="bold", ha="center", va="center", zorder=7)
dec_handles = [Line2D([0],[0], color=DEC_COLORS[i], lw=2.2, label=f"{d0}–{d1}")
               for i,(d0,d1) in enumerate(DECADES)]
ref_handles = [Line2D([0],[0], color=c, lw=2.8, label=lab) for _, c, lab in REF]
sea_handles = [Line2D([0],[0], color="0.3", lw=2, ls="-",  label="March (winter max)"),
               Line2D([0],[0], color="0.3", lw=2, ls="--", label="September (summer min)")]
leg1 = ax.legend(handles=dec_handles + ref_handles, title="Mean sea-ice edge (15% SIC)",
                 loc="lower right", fontsize=7.5, title_fontsize=8)
ax.add_artist(leg1); ax.legend(handles=sea_handles, loc="lower left", fontsize=7.5)
ax.set_title("DBO sea-ice edge (15% SIC): decadal means + reference years 2012 and this year",
             fontsize=12, loc="left", fontweight="bold")
plt.tight_layout(); plt.show()


  decadal: 2806 days to stream
    decadal 200/2806
    decadal 400/2806
    decadal 1000/2806
    decadal 1200/2806
    decadal 1400/2806
    decadal 1600/2806
    decadal 2000/2806
    decadal 2200/2806
    decadal 2400/2806
    decadal 2600/2806
    decadal 2800/2806
  ref: 122 days to stream
saved -> ~/dbo_edge_maps.pkl


## 6. Streaming pass (parallel, memory-bounded, checkpointed) — build everything

Builds `box_ts` (daily ice fraction per box), `mean_map` (record-mean), and `annual_ice` (per-year
mean). Checkpoints every `CKPT_EVERY` days so a crash resumes.

In [7]:
import pickle
def _have_products():
    return all(k in globals() and globals()[k] is not None for k in ("box_ts","mean_map","annual_ice"))
ny, nx = dbo_area.shape

if CACHE_DERIVED and CACHE_PATH.exists() and not _have_products():
    _d = pickle.load(open(CACHE_PATH, "rb"))
    box_ts, mean_map, annual_ice = _d["box_ts"], _d["mean_map"], _d["annual_ice"]
    print("loaded finished products — no streaming.")

if _have_products():
    print("products already in memory — skipping the stream.")
else:
    if CHECKPOINT and CKPT_PATH.exists():
        _ck = pickle.load(open(CKPT_PATH, "rb"))
        mean_sum, mean_cnt, ann, box_rows = _ck["mean_sum"], _ck["mean_cnt"], _ck["ann"], _ck["box_rows"]
        print(f"resumed: {len(box_rows)} days already folded.")
    else:
        mean_sum = np.zeros((ny, nx), np.float32); mean_cnt = np.zeros((ny, nx), np.float32)
        ann = {}; box_rows = {}

    def _stream_one(item):
        d, g = item
        try: return d, project_day(g, dbo_area), None
        except Exception as e: return d, None, f"{type(e).__name__}: {str(e)[:70]}"

    def _fold(d, arr):
        m = np.isfinite(arr)
        mean_sum[m] += arr[m]; mean_cnt[m] += 1
        s = ann.setdefault(d.year, [np.zeros((ny, nx), np.float32), np.zeros((ny, nx), np.float32)])
        s[0][m] += arr[m]; s[1][m] += 1
        bm = {}
        for n, mask in box_masks.items():
            v = arr[mask & m]; bm[n] = float(np.mean(v)) if v.size else np.nan
        box_rows[d] = bm

    def _save_ckpt():
        if not CHECKPOINT: return
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        tmp = CKPT_PATH.with_suffix(".tmp")
        pickle.dump({"mean_sum": mean_sum, "mean_cnt": mean_cnt, "ann": ann, "box_rows": box_rows}, open(tmp, "wb"))
        tmp.replace(CKPT_PATH)

    remaining = [(d, g) for d, g in sorted(granules_by_day.items()) if d not in box_rows]
    src = iter(remaining); MAXQ = max(2, N_WORKERS*2); t0 = _time.time(); n_ok = n_fail = since = 0
    print(f"{len(box_rows)} done | streaming {len(remaining)} more...")
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        inflight = set()
        for _ in range(MAXQ):
            try: inflight.add(ex.submit(_stream_one, next(src)))
            except StopIteration: break
        while inflight:
            done, inflight = wait(inflight, return_when=FIRST_COMPLETED)
            for fut in done:
                d, arr, err = fut.result()
                if arr is None:
                    n_fail += 1
                    if n_fail <= 10: print(f"  {d}  X {err}")
                else:
                    _fold(d, arr); n_ok += 1; since += 1
                    if n_ok % 500 == 0: print(f"  {n_ok} ok / {n_fail} failed  ({(_time.time()-t0)/60:.1f} min)")
                    if since >= CKPT_EVERY: _save_ckpt(); since = 0
                try: inflight.add(ex.submit(_stream_one, next(src)))
                except StopIteration: pass
            del done
    _save_ckpt()

    dates = sorted(box_rows); tcoord = np.array([np.datetime64(d.isoformat()) for d in dates])
    box_ts = {n: xr.DataArray(np.array([box_rows[d][n] for d in dates]), dims="time", coords={"time": tcoord})
              for n in box_masks}
    mean_map = np.where(mean_cnt > 0, mean_sum / mean_cnt, np.nan)
    _years = sorted(ann)
    annual_ice = xr.DataArray(np.stack([np.where(ann[y][1] > 0, ann[y][0]/ann[y][1], np.nan) for y in _years]),
                              dims=("year","y","x"), coords={"year": _years})
    gc.collect()
    print(f"\nfolded {len(box_rows)} days -> box_ts, mean_map, {len(_years)} annual frames.")

if CACHE_DERIVED and not CACHE_PATH.exists():
    pickle.dump({"box_ts": box_ts, "mean_map": mean_map, "annual_ice": annual_ice}, open(CACHE_PATH, "wb"))
    print("saved finished products.")

resumed: 1000 days already folded.
1000 done | streaming 15802 more...
  500 ok / 0 failed  (0.3 min)
  1000 ok / 0 failed  (0.6 min)
  1500 ok / 0 failed  (1.8 min)
  2000 ok / 0 failed  (3.2 min)
  2500 ok / 0 failed  (5.4 min)
  3000 ok / 0 failed  (7.6 min)
  3500 ok / 0 failed  (9.7 min)
  4000 ok / 0 failed  (11.9 min)
  4500 ok / 0 failed  (14.1 min)
  5000 ok / 0 failed  (16.2 min)
  5500 ok / 0 failed  (18.4 min)
  6000 ok / 0 failed  (20.5 min)
  6500 ok / 0 failed  (22.7 min)
  7000 ok / 0 failed  (24.8 min)
  7500 ok / 0 failed  (27.0 min)
  8000 ok / 0 failed  (29.1 min)
  8500 ok / 0 failed  (31.3 min)
  9000 ok / 0 failed  (33.4 min)
  9500 ok / 0 failed  (35.6 min)
  10000 ok / 0 failed  (37.7 min)
  10500 ok / 0 failed  (39.8 min)
  11000 ok / 0 failed  (42.0 min)
  11500 ok / 0 failed  (44.2 min)
  12000 ok / 0 failed  (46.3 min)
  12500 ok / 0 failed  (48.5 min)
  13000 ok / 0 failed  (50.6 min)
  13500 ok / 0 failed  (52.7 min)
  14000 ok / 0 failed  (54.9 min)
  14

---

# Part B — NSIDC-0747 day of retreat

*This part was a separate notebook before the repository was consolidated; it starts from its own imports and configuration, so it can be run on its own after the sections above.*

# NSIDC-0747 **sea-ice retreat timing** over DBO — streaming, EASE-Grid 2.0 North (no storage)

Companion to `07_sea-ice-ssmi-and-nsidc0747.ipynb`. Instead of computing melt/freeze phenology from daily SIC, this
uses the **NSIDC-0747 v2 Arctic Sea Ice Seasonal Change and Melt/Freeze Climate Indicators**
product, which *already provides* the **day of retreat (DOR)** — the last day sea-ice concentration
drops below 15% each ice-year — plus advance, opening, closing and the open-water period, on a
**per-year** basis for **1979–2023**. We stream the single netCDF from the NSIDC DAAC, regrid every
field onto the **same EASE-2 4 km DBO grid** used in the SSM/I notebook, save a **pickle**, and make
the retreat-timing figures (per-box trends, trend map, anomaly GIF).

### Product
- **`NSIDC-0747`** v2 (DOI 10.5067/2ZJKKYXYLYZ0), Steele/Bliss/Peng/Meier/Dickinson. A **single**
  netCDF file `arctic_seaice_climate_indicators_nh_v01r01_1979-2024.nc`, NSIDC north polar-stereo
  (EPSG:3411) **25 km**, grid **448 × 304**, dim order `(time, y, x)` with `time` = start of each ice
  year (1 Mar 1979 → 28 Feb 2024, so ice-years **1979–2023**).
- Key variables (all **day-of-year**): **`DOR`** day of retreat (SIC last <15%), **`DOA`** day of
  advance (SIC first >15%), **`DOO`** opening (SIC last <80%), **`DOC`** closing (SIC first >80%).
  Period variables (**days**): **`IIFP`** inner ice-free / open-water period (`DOA-DOR`), `OIFP`,
  `SLIP`, `SGIP`. 2-D `latitude`/`longitude` geolocation is included.
- **Earlier retreat = negative trend (days/decade) = longer open-water season.** 1984 DOO/SLIP/OIFP
  are flagged missing in the source (insufficient spring-1984 SIC).

### Streaming
Streams the one file via `earthaccess` (nothing written but the derived **pickle**). Everything after
the stream reads from the pickle, so re-running plots costs nothing.

## 1. Imports

In [16]:
import time as _time, warnings, gc, pickle
from pathlib import Path

import numpy as np, pandas as pd, xarray as xr
import earthaccess
from pyresample import AreaDefinition, geometry
from pyresample.kd_tree import resample_nearest
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import cartopy.crs as ccrs, cartopy.feature as cfeature
from scipy.stats import theilslopes, kendalltau, norm
warnings.filterwarnings("ignore")
print("imports OK")

imports OK


## 1b. Figure style — Nature / Science house style

Sans-serif, thin recessive axes (no top/right spines), small ticks, no chartjunk; a fixed
colorblind-safe **Okabe-Ito** categorical order; a single-hue sequential ramp and a
gray-midpoint diverging ramp. Applied globally via `rcParams` so every figure inherits it.

In [17]:
import matplotlib as mpl
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "figure.facecolor": "white", "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Helvetica Neue", "Arial", "Nimbus Sans", "DejaVu Sans"],
    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "axes.edgecolor": "0.2", "axes.labelcolor": "0.0",
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlepad": 4,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.size": 3, "ytick.major.size": 3, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.color": "0.2", "ytick.color": "0.2",
    "legend.frameon": False, "axes.grid": False, "grid.linewidth": 0.4, "grid.color": "0.85",
    "lines.linewidth": 1.3, "lines.markersize": 4, "lines.solid_capstyle": "round",
})

# Okabe-Ito colorblind-safe categorical order (fixed; never cycled)
OI = {"blue":"#0072B2", "green":"#009E73", "orange":"#D55E00", "purple":"#CC79A7",
      "sky":"#56B4E9", "amber":"#E69F00", "yellow":"#F0E442", "ink":"#222222"}
ACCENT, TRENDLINE = OI["blue"], OI["orange"]

def panel_letter(ax, letter, dx=-0.02, dy=1.04):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom", ha="right")
import string
print("Nature/Science style applied")

Nature/Science style applied


## 2. Configuration — product, EASE-2 grid, DBO boxes

The EASE-2 `dbo_area`, `DBO_BOXES` and display projections are **identical** to `07_sea-ice-ssmi-and-nsidc0747.ipynb`
so the maps line up box-for-box.

In [18]:
# --- NSIDC-0747 v2 (single netCDF, 1979-2024 ice-years) ---
ICE_SHORT   = "NSIDC-0747"
ICE_VERSION = "2"
ICE_CONCEPT = "C4085455832-NSIDCV0"      # NSIDC-0747 v2; pins version
LATN, LONN  = "latitude", "longitude"

# retreat-timing variables to regrid + keep (day-of-year unless noted). IIFP is a period (days).
KEEP_VARS = ["DOR", "DOA", "DOO", "DOC", "IIFP"]
LAT_MIN, LAT_MAX = 45, 90
ROI = 30_000                              # nearest-neighbour radius (m); native 25 km

# EASE-Grid 2.0 North subset, 4 km  (SAME as the SSM/I notebook)
dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000))

# derived-products pickle (the only thing written)
CACHE_PATH = Path.home() / "nsidc0747_retreat_products.pkl"  # add your path

DBO_BOXES = {
    "DBO1": [(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
    "DBO2": [(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
    "DBO3": [(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
    "DBO4": [(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
    "DBO5": [(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
    "DBO6": [(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
    "DBO7": [(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
    "DBO8": [(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)],
}
data_crs = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=0)
disp     = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=-152)
x0, y0, x1, y1 = dbo_area.area_extent
MAP_EXTENT = [-180, -124, 49, 73.5]        # same framing as the SSM/I maps

# colormaps: single-hue sequential for absolute day-of-retreat, gray-midpoint diverging for trends
try:
    import cmocean; DAY_CMAP = cmocean.cm.dense       # perceptual single-hue sequential
except ImportError:
    DAY_CMAP = "cividis"                              # colorblind-safe sequential fallback
TREND_CMAP = "coolwarm"                               # blue = earlier retreat (ice loss), red = later
BOX_COLOR  = "#c1272d"                                # DBO box outlines (matches existing figures)
print(f"{ICE_SHORT} v{ICE_VERSION} | keep {KEEP_VARS} | pickle -> {CACHE_PATH}")

NSIDC-0747 v2 | keep ['DOR', 'DOA', 'DOO', 'DOC', 'IIFP'] | pickle -> ~/nsidc0747_retreat_products.pkl


## 3. Auth + open the single file (direct NSIDC HTTPS)

**NSIDC-0747 is *not* granule-indexed in CMR** — `earthaccess.search_data` returns nothing, and the
exact filename in the DAAC directory isn't advertised in the metadata. So we **crawl the
Earthdata-authenticated directory index**, find the `.nc` file (descending one subdirectory level if
needed), and download it once to a local cache (the file is only a few MB). Needs a `~/.netrc` with
your Earthdata login — `earthaccess.login(persist=True)` writes one if missing.

In [19]:
import re
auth = earthaccess.login(persist=True)

NSIDC_DIR = "https://daacdata.apps.nsidc.org/pub/DATASETS/nsidc0747_seaice_melt_indicators_v2/"

def find_nc_url(base, session):
    """Return the URL of the first .nc file in `base` (or one subdir down)."""
    def hrefs(url):
        r = session.get(url, timeout=60); r.raise_for_status()
        return re.findall(r'href=["\']([^"\'?][^"\']*)', r.text)
    top = hrefs(base)
    ncs = [h.rsplit('/', 1)[-1] for h in top if h.lower().endswith(".nc")]
    if ncs: return base + ncs[0]
    for h in top:                                   # descend into subdirectories
        if h.endswith("/") and not h.startswith((".", "/", "?")):
            sub = base + h
            try:
                sncs = [x.rsplit('/', 1)[-1] for x in hrefs(sub) if x.lower().endswith(".nc")]
                if sncs: return sub + sncs[0]
            except Exception: continue
    raise FileNotFoundError(f"no .nc under {base}")

_session = earthaccess.get_requests_https_session()
NSIDC_URL = find_nc_url(NSIDC_DIR, _session)
NC_CACHE  = Path.home() / NSIDC_URL.rsplit('/', 1)[-1]  # add your path
print("found:", NSIDC_URL)

def open_nsidc0747():
    """Open the NSIDC-0747 netCDF (Earthdata-auth cached download)."""
    if not NC_CACHE.exists():
        with _session.get(NSIDC_URL, stream=True, timeout=300) as r:
            r.raise_for_status()
            tmp = NC_CACHE.with_suffix(".tmp")
            with open(tmp, "wb") as fh:
                for chunk in r.iter_content(1 << 20): fh.write(chunk)
            tmp.replace(NC_CACHE)
        print(f"  cached -> {NC_CACHE} ({NC_CACHE.stat().st_size/1e6:.1f} MB)")
    return xr.open_dataset(NC_CACHE, engine="h5netcdf")

def ice_years(ds):
    """Integer ice-year for each time step. The file's time uses a Julian calendar
    (decoded to cftime), so go through xarray's .dt accessor, not pd.to_datetime."""
    t = ds["time"]
    try:
        return np.asarray(t.dt.year).astype(int)
    except Exception:
        vals = np.atleast_1d(t.values)
        return np.array([getattr(v, "year", pd.Timestamp(v).year) for v in vals], int)

with open_nsidc0747() as _ds:
    print(f"OK - dims {dict(_ds.sizes)} | {len(_ds.data_vars)} variables | "
          f"ice-years {ice_years(_ds).min()}-{ice_years(_ds).max()}")

found: https://daacdata.apps.nsidc.org/pub/DATASETS/nsidc0747_seaice_melt_indicators_v2/arctic_seaice_climate_indicators_nh_v02r00_1979-2024.nc
OK - dims {'time': 45, 'y': 448, 'x': 304} | 15 variables | ice-years 1979-2023


## 4. Stream + regrid onto EASE-2 DBO grid -> products, save pickle

Reads the one file and nearest-neighbour regrids each kept variable to `(year, y, x)` on the EASE-2
grid, then pickles the result. Day-of-year values <=0 or >500 are treated as missing. Crucially, each
slice is **masked to valid cells before resampling**, so a coastal DBO cell borrows the nearest
valid (open-water) 25 km value within `ROI` instead of the land-masked NaN cell directly beneath it —
otherwise the shallow shelf boxes (DBO1-7) come back mostly empty.

In [21]:
def build_products():
    ds = open_nsidc0747()                                 # mask_and_scale=True -> _FillValue = NaN
    years = ice_years(ds)
    lon_s = np.asarray(ds[LONN].values, float); lat_s = np.asarray(ds[LATN].values, float)
    lon_s = np.where(lon_s > 180, lon_s - 360, lon_s)
    geo_ok = np.isfinite(lat_s) & np.isfinite(lon_s) & (lat_s >= LAT_MIN) & (lat_s <= LAT_MAX)
    empty = np.full(dbo_area.shape, np.nan, np.float32)

    def regrid(a2d):
        # mask to VALID source cells BEFORE building the swath, so a coastal DBO cell
        # snaps to the nearest valid (open-water) 25 km value within ROI instead of a
        # land-masked NaN cell sitting directly under it.
        a = np.asarray(a2d, float)
        a = np.where((a > 0) & (a < 500), a, np.nan)     # drop fill / out-of-range
        m = geo_ok & np.isfinite(a)
        if not m.any(): return empty.copy()
        swath = geometry.SwathDefinition(lons=lon_s[m], lats=lat_s[m])
        g = resample_nearest(swath, a[m], dbo_area, radius_of_influence=ROI,
                             fill_value=np.nan, reduce_data=False)
        return g.astype(np.float32)

    prod = {}
    for v in [v for v in KEEP_VARS if v in ds.data_vars]:
        stack = np.stack([regrid(ds[v].isel(time=k).values) for k in range(ds.sizes["time"])])
        prod[v] = xr.DataArray(stack, dims=("year", "y", "x"), coords={"year": years})
        cov = np.isfinite(stack).any(0).mean() * 100
        print(f"  regridded {v}: {stack.shape} | {cov:.0f}% of grid ever valid")
    ds.close()
    return {"prod": prod, "years": years}

if CACHE_PATH.exists():
    _d = pickle.load(open(CACHE_PATH, "rb"))
    prod, years = _d["prod"], _d["years"]
    print(f"loaded products from {CACHE_PATH.name} (no re-stream) | years {years.min()}-{years.max()}")
else:
    _d = build_products(); prod, years = _d["prod"], _d["years"]
    tmp = CACHE_PATH.with_suffix(".tmp")
    pickle.dump(_d, open(tmp, "wb")); tmp.replace(CACHE_PATH)
    print(f"saved -> {CACHE_PATH}  ({CACHE_PATH.stat().st_size/1e6:.1f} MB)")
gc.collect()

loaded products from nsidc0747_retreat_products.pkl (no re-stream) | years 1979-2023


516